<a href="https://colab.research.google.com/github/Chanaka3534/FYP_XGBOOST-Using-past-data/blob/2.0/FYP_XGBOOST(Use_past_days_data).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 🚀 Load data
df = pd.read_csv("FYP_DATA.csv")

# 🧼 Clean numeric columns
for col in ["Resovior water level(m)", "Resovior discharge rate", "Water level(Kaliodai)"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "").replace("-", np.nan))

# ❌ Drop missing values
df.dropna(inplace=True)

# 🧠 Encode target
le = LabelEncoder()
df["Flood risk"] = le.fit_transform(df["Flood risk"])

# 🗓️ Sort by date to maintain time order
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

# 🔢 Define input features
features = ["Catchment Rainfall", "Downstream rainfall", "Resovior water level(m)",
            "Resovior discharge rate", "Water level(Kaliodai)"]

# 🔁 Create sequences
X = []
y = []
sequence_length = 30  # number of past days to use

for i in range(len(df) - sequence_length):
    X_seq = df[features].iloc[i:i+sequence_length].values.flatten()
    y_seq = df["Flood risk"].iloc[i + sequence_length]
    X.append(X_seq)
    y.append(y_seq)

X = np.array(X)
y = np.array(y)

# 🔀 Split chronologically (first 80% train, last 20% test)
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]


# ✅ Apply SMOTE with k_neighbors=1
sm = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)


# ✅ Train XGBoost model
model = XGBClassifier(eval_metric="mlogloss")
model.fit(X_train_resampled, y_train_resampled)

# ✅ Predict and evaluate
y_pred = model.predict(X_test)

# 📊 Results
accuracy = accuracy_score(y_test, y_pred)
f1_per_class = f1_score(y_test, y_pred, average=None)
class_names = le.classes_

print(f"\n✅ Accuracy: {accuracy:.4f}")
print("✅ F1 Score per class:")
for label, score in zip(class_names, f1_per_class):
    print(f"  {label}: {score:.4f}")



✅ Accuracy: 0.9774
✅ F1 Score per class:
  NEGATIVE: 0.9885
  POSITIVE: 0.3636


In [ ]:
# ✅  30-day manual input (list of 30 rows)
manual_data = [
    [6.7, 0, 103.2, 1700, 1200],
    [9.1, 0, 103.2, 1700, 1200],
    [10, 10.5, 103.1, 1700, 900],
    [9.5, 0, 103.1, 1500, 900],
    [16.2, 0, 103, 1700, 800],
    [13.6, 28.4, 102.9, 1600, 800],
    [6.3, 16, 102.8, 1600, 800],
    [13.6, 3.2, 102.8, 400, 1500],
    [1.7, 0, 102.9, 500, 1500],
    [1.3, 29.3, 103.1, 500, 1500],
    [14.1, 0, 103.3, 300, 1500],
    [0.5, 0, 103.5, 1100, 1500],
    [2.5, 0, 103.5, 750, 1500],
    [3.7, 0, 103.4, 750, 1500],
    [13.5, 0, 103.4, 1050, 1200],
    [4.7, 0, 103.8, 1150, 1200],
    [4.9, 0, 103.9, 1600, 900],
    [0.4, 0, 103.9, 1600, 900],
    [2.1, 2, 103.9, 1800, 900],
    [2.6, 0, 103.9, 1500, 700],
    [1.1, 0, 103.9, 1600, 700],
    [7, 5.3, 103.9, 1600, 1100],
    [8.6, 0, 104.1, 1250, 2500],
    [2.7, 0, 104.2, 1150, 2500],
    [5.4, 0, 104.4, 1150, 2000],
    [13.3, 0, 104.4, 1250, 2000],
    [1.7, 0, 104.5, 1250, 2300],
    [15.3, 0, 104.6, 1250, 2300],
    [4, 0, 104.8, 900, 2300],
    [15.1, 22, 105.1, 0, 2300],
]

# ✅ Convert rows to columns: separate lists
import numpy as np

manual_data_array = np.array(manual_data)

# Now split columns:
catchment_rainfall_30 = manual_data_array[:, 0].tolist()
downstream_rainfall_30 = manual_data_array[:, 1].tolist()
reservoir_level_30 = manual_data_array[:, 2].tolist()
discharge_rate_30 = manual_data_array[:, 3].tolist()
kaliodai_level_30 = manual_data_array[:, 4].tolist()

# Combine all 5 in order:
manual_input = (
    catchment_rainfall_30
    + downstream_rainfall_30
    + reservoir_level_30
    + discharge_rate_30
    + kaliodai_level_30
)

print(f"Manual input length: {len(manual_input)}")

# ✅ Now predict:
manual_pred = model.predict([manual_input])
print("Predicted Flood Risk for tomorrow:", le.inverse_transform(manual_pred)[0])

Manual input length: 150
Predicted Flood Risk for tomorrow: NEGATIVE
